In [ ]:
# install and download spaCy related modules
!pip install --upgrade spacy
!python -m spacy download en_core_web_sm

# spaCy
import spacy
from spacy.language import Language
from spacy.tokens import Span
from spacy.matcher import PhraseMatcher

# Google Drive
from google.colab import drive

# Firebase/Firestore
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

# general Python modules
import json
from datetime import datetime
from pprint import pprint


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
/usr/local/lib/python3.8/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2023-02-13 20:04:07.605244: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2023-02-13 20:04:07.605406: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2023-02-13 20:04:07.605432: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries.

In [ ]:
import requests
from bs4 import BeautifulSoup as BS
import spacy
import json
import random
import re

nlp = spacy.load("en_core_web_sm")

api = "https://www.wikidata.org/w/api.php"

In [ ]:
# remount drive, forced if needed
drive.mount("/content/gdrive/", force_remount = True)
print("Established access to Google Drive")

# initialize Drive path
DRIVE_PATH = "/content/gdrive/My Drive"

Mounted at /content/gdrive/
Established access to Google Drive


In [ ]:
# Creating list of 500 entities

response = requests.get(
	url="https://en.wikipedia.org/wiki/Cryptocurrency",
)
soup = BS(response.content, 'html.parser')

# Getting all the links
allLinks = soup.find(id="bodyContent").find_all("a")
allentities = []
# Creating a list of ignore_entities which are not present on Wikidata
ignore_entities=[]
for link in allLinks:
    if link.get('title')== None or link['href'].find("/wiki/")==-1:
      continue
    x = link.get('title')
    if ("(" in x or ":" in x or "-" in x):
      ignore_entities.append(x)
    else:
      if x not in allentities:
        allentities.append(x)
    
entities_list=random.choices(allentities,k=500)
print(entities_list)
print(len(entities_list))
print(ignore_entities)
print(len(ignore_entities))

['Jihan Wu', 'Oaktree Capital Management', 'Market capitalization', 'Central bank digital currency', 'History of cryptography', 'Craig Steven Wright', 'Cryptoeconomics', 'Aventus Protocol', 'Secure by design', 'Liberty Lobby', 'Purchasing power', 'OneCoin', 'Bailment', 'Stream cipher', 'Decryption', 'Ancient Mesopotamian units of measurement', 'Agustín Carstens', 'Decentralized finance', 'Housing market bubble', 'Richard B. Spencer', 'Federal Reserve System', 'Cryptocurrency exchange', '2020 Twitter account hijacking', 'Cryptographic nonce', 'Wei Dai', 'Encryption', 'Baidu', 'List of bitcoin companies', 'U.S. Securities and Exchange Commission', 'SafeMoon', 'Vitalik Buterin', 'Cypherpunk', 'Bank for International Settlements', 'Smart contract', 'Proof of authority', 'China Central Bank', 'BlackRock', 'Economics of bitcoin', 'National Bureau of Economic Research', 'Electricity market', 'Tencent', 'Basel Committee on Banking Supervision', 'Tobias Adrian', 'Angus Deaton', 'Mining pool', '

# Gazeteer list and entities list

In [ ]:
from ast import Div
list = []
gazeteer_list = []

for i in range (0, len(entities_list)-1):
  # print(i)
  query = entities_list[i]
  params = {
    "action" : "wbsearchentities",
    "language" : "en",
    "format" : "json",
    "search" : query
  }
  r = requests.get(api, params = params)
  
  try:
    id = r.json()['search'][0]['id']
    url = "https://www.wikidata.org/wiki/" + id
    result = requests.get(url)
    doc = BS(result.text, "html.parser")

  # retrieving Aliases 
    alias = doc.find_all( ["li"],class_= "wikibase-entitytermsview-aliases-alias")
    # print(alias)
    alias_list = []
    for i in range(0, len(alias)):
      alias_list.append(alias[i].string)

  # Retrieving description
    des=doc.find_all("div",class_="wikibase-entitytermsview-heading-description")
    for res in des:
      description=res.text

  # Retrieving Label 
    doc = nlp(r.json()['search'][0]['label'])
    ents = [(e.label_) for e in doc.ents]
    if len(ents) == 0:
      ans = ""
    else:
      ans=ents[0] 

  # appending in the entities_list
    list.append({
          'qid' : r.json()['search'][0]['id'],
          'name' : r.json()['search'][0]['label'],
          'label' : ans,
        # 'description' : r.json()['search'][0]['description'],
          'description' : description,
          'aliases' : alias_list
        
      })
    
  # appending entities and aliases in the gazeteer_list
    gazeteer_list.append(entities_list[i])
    for i in range(0, len(alias)):
      gazeteer_list.append(alias[i].string)
      

  except:
    pass

print("Both lists created successfully")

Both lists created successfully


**Saving the files as json in drive**

In [ ]:
with open('/content/gdrive/My Drive/ie_course_2022_team08/retrieved_data/entities_list.json', 'w', encoding='utf-8') as file:
  json.dump(list, file, indent=4, ensure_ascii='False', default=str)


with open('/content/gdrive/My Drive/ie_course_2022_team08/retrieved_data/gazeteer_list2.json', 'w', encoding='utf-8') as file:
  json.dump(gazeteer_list, file, indent=4, ensure_ascii='False', default=str)

with open('/content/gdrive/My Drive/ie_course_2022_team08/retrieved_data/entities_gazetteers_ignore_list.json', 'w', encoding='utf-8') as file:
  json.dump(ignore_entities, file, indent=4, ensure_ascii='False', default=str)

In [ ]:

# retrieve list of entities gazetteers
with open(DRIVE_PATH + "/ie_course_2022_team08/retrieved_data/gazeteer_list2.json") as f:
  entities_gazetteers_list = json.load(f)
  print("Retrieved entities gazetteers list")
  
# retrieve ignore list of entities gazetteers
with open(DRIVE_PATH + "/ie_course_2022_team08/retrieved_data/entities_gazetteers_ignore_list.json") as f:
  entities_gazetteers_ignore_list = json.load(f)
  print("Retrieved entities gazetteers' ignore list")

# retrieve entities
with open(DRIVE_PATH + "/ie_course_2022_team08/retrieved_data/entities_list.json") as f:
  entities = json.load(f)
  print("Retrieved entities")

Retrieved entities gazetteers list
Retrieved entities gazetteers' ignore list
Retrieved entities


In [ ]:
""" Custom pipeline Component: entities gazetteer function """

@Language.component("rule_based")
def rule_based(doc):
  # set up and extend structure of default span object
  span = Span(doc, 0, 0, "")
  span.set_extension("qid", default=None, force=True)
  span.set_extension("label", default=None, force=True)
  span.set_extension("wd_name", default=None, force=True)

  # identify matches of the gazetteers contained in Doc object (text)
  matches = matcher(doc)
  # convert matches to Span objects
  spans = [doc[start:end] for _, start, end in matches]
  # filter overlaping matches (Span objs) to keep gazetteers uniqueness
  filtered_matches = spacy.util.filter_spans(spans)

  # loop unique matches of gazetteers
  for match in filtered_matches:
    # skip if matched gazetter is in ignore list
    if match.text in entities_gazetteers_ignore_list:
      print(f"-- Skipped '{match}' due it's in ignore list!")
      continue
    # find matched gazetters in issues dictionary to get entities' Wikidata info
    # usually only one entity is found, but some gazetteer finds more than one
    matched_entities = [i for i in entities if match.text == i["name"] or match.text in i["aliases"]]
    if len(matched_entities):
      entity = Span(doc, match.start, match.end, label=matched_entities[0]["label"])

      # set attributes
      if len(matched_entities) == 1:
        entity._.label = matched_entities[0]["label"]
      elif len(matched_entities) > 1:
        entity._.label = [e["label"] for e in matched_entities]

      entity.set_extension("qid", default=None, force=True)
      if len(matched_entities) == 1:
        entity._.qid = matched_entities[0]["qid"]
      elif len(matched_entities) > 1:
        entity._.qid = [e["qid"] for e in matched_entities]

      entity.set_extension("wd_name", default=None, force=True)
      if len(matched_entities) == 1:
        entity._.wd_name = matched_entities[0]["name"]
      elif len(matched_entities) > 1:
        entity._.wd_name = [e["name"] for e in matched_entities]

      # modify the provided entity spans, leaving the rest unmodified
      doc.set_ents([entity], default="unmodified")

  return doc

# create pipeline loaded with a pretrained statistical model (English/lg)
nlp = spacy.load("en_core_web_lg", exclude=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"])
nlp.add_pipe("sentencizer")

# add custom component to pipeline
nlp.add_pipe("rule_based", last=True)

# initialize spaCY phrase matcher (rule-based)
matcher = PhraseMatcher(nlp.vocab, None)

# load gazetteers (issues) as matcher patterns
patterns = [nlp.make_doc(gazetteer) for gazetteer in entities_gazetteers_list]
matcher.add("gazetteers", patterns)

# see pipeline components
print(nlp.pipe_names)

# analize pipeline
pprint(nlp.analyze_pipes(pretty=True))

rule_nlp_dir = DRIVE_PATH + "/ie_course_2022_team08/output/ml_el/rule_based_nlp"

nlp.to_disk("rule_nlp_dir")

['ner', 'sentencizer', 'rule_based']

============================= Pipeline Overview =============================

#   Component     Assigns               Requires   Scores          Retokenizes
-   -----------   -------------------   --------   -------------   -----------
0   ner           doc.ents                         ents_f          False      
                  token.ent_iob                    ents_p                     
                  token.ent_type                   ents_r                     
                                                   ents_per_type              
                                                                              
1   sentencizer   token.is_sent_start              sents_f         False      
                  doc.sents                        sents_p                    
                                                   sents_r                    
                                                                              
2   rule_based

In [ ]:
# Packaging this custom component model
# So that we can load it in other python scripts
# We will get .tag.gz file which we can use to load this model

!python -m spacy package rule_nlp_dir compiled2 --code my_component.py

/usr/local/lib/python3.8/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2023-02-13 20:07:12.628402: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2023-02-13 20:07:12.628515: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2023-02-13 20:07:12.628534: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nvidia GPU with TensorRT, please make sure the missing libraries mentione

In [ ]:
sentence = "Many of the scammers behind Bitcoin and MobileCoin ATM tokens use crypto-to-fiat exchanges to both seed their scams and launder their proceeds. PPCoin is also known as Peercoin and PPC. Blake is the one who is the founder of cryptocurrencies in USD. Federal Reserve System is the central bank for digital currencies. Also known as Federal reserve."
doc = nlp(sentence)
for ent in doc.ents:
  print(ent.text, ent.start_char, ent.end_char, ent.label_, ent._.qid, ent._.wd_name)

MobileCoin 40 50 PRODUCT None None
PPCoin 144 150 NORP Q15177466 Peercoin
Peercoin 168 176 NORP None None
PPC 181 184 NORP Q15177466 Peercoin
Blake 186 191 PERSON None None
USD 245 248 GPE ['Q4917', 'Q4917'] ['United States dollar', 'United States dollar']
Federal Reserve System 250 272 ORG None None
Federal reserve 331 346 ORG None None
